# Style-Bert-VITS2 (ver 2.7.0) のGoogle Colabでの学習

Google Colab上でStyle-Bert-VITS2の学習を行うことができます。

このnotebookでは、通常使用ではあなたのGoogle Driveにフォルダ`Style-Bert-VITS2`を作り、その内部での作業を行います。他のフォルダには触れません。
Google Driveを使わない場合は、初期設定のところで適切なパスを指定してください。

## 流れ

### 学習を最初からやりたいとき
上から順に実行していけばいいです。音声合成に必要なファイルはGoogle Driveの`Style-Bert-VITS2/model_assets/`に保存されます。また、途中経過も`Style-Bert-VITS2/Data/`に保存されるので、学習を中断したり、途中から再開することもできます。

### 学習を途中から再開したいとき
0と1を行い、3の前処理は飛ばして、4から始めてください。スタイル分け5は、学習が終わったら必要なら行ってください。


## 0. 環境構築

Style-Bert-VITS2の環境をcolab上に構築します。ランタイムがT4等のGPUバックエンドになっていることを確認し、実行してください。

**注意**: このセルを実行した後に「セッションがクラッシュしました」「不明な理由により、セッションがクラッシュしました。」等の警告が出ますが、**無視してそのまま先へ**進んでください。（一度ランタイムを再起動させてnumpy<2を強制させるため `exit()` を呼んでいることからの措置です。）

In [1]:
import os

os.environ["PATH"] += ":/root/.cargo/bin"

!curl -LsSf https://astral.sh/uv/install.sh | sh
!git clone https://github.com/litagin02/Style-Bert-VITS2.git
%cd Style-Bert-VITS2/
!uv pip install --system -r requirements-colab.txt --no-progress
!python initialize.py --skip_default_models

# 互換性のあるバージョンセットをインストール
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0
!pip install transformers==4.35.2

exit()

downloading uv 0.9.26 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Cloning into 'Style-Bert-VITS2'...
remote: Enumerating objects: 7075, done.
remote: Counting objects: 100% (3871/3871), done.
remote: Compressing objects: 100% (1114/1114), done.
remote: Total 7075 (delta 2948), reused 2757 (delta 2757), pack-reused 3204 (from 3)
Receiving objects: 100% (7075/7075), 16.66 MiB | 16.49 MiB/s, done.
Resolving deltas: 100% (4712/4712), done.
/content/Style-Bert-VITS2
Using Python 3.12.12 environment at: /usr
Resolved 178 packages in 3.49s
Prepared 46 packages in 5.64s
Uninstalled 3 packages in 75ms
Installed 46 packages in 528ms
 + asteroid-filterbanks==0.4.0
 + cmudict==1.1.3
 + cn2an==0.5.23
 + coloredlogs==15.0.1
 + colorlog==6.10.1
 + distance==0.1.3
 + docopt==0.6.2
 + g2p-en==2.1.0
 + humanfriendly==10.0
 + hyperpyyaml==1.2.3
 + julius==0.2.7
 - librosa==0.11.0
 + librosa==0.9.2
 + lightning==2.6.0
 + lightning-utiliti

In [1]:
# Google driveを使う方はこちらを実行してください。

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 1. 初期設定

学習とその結果を保存するディレクトリ名を指定します。
Google driveの場合はそのまま実行、カスタマイズしたい方は変更して実行してください。

In [2]:
# 作業ディレクトリを移動
%cd /content/Style-Bert-VITS2/

# 学習に必要なファイルや途中経過が保存されるディレクトリ
dataset_root = "/content/drive/MyDrive/Style-Bert-VITS2/Data"

# 学習結果（音声合成に必要なファイルたち）が保存されるディレクトリ
assets_root = "/content/drive/MyDrive/Style-Bert-VITS2/model_assets"

import yaml


with open("configs/paths.yml", "w", encoding="utf-8") as f:
    yaml.dump({"dataset_root": dataset_root, "assets_root": assets_root}, f)

/content/Style-Bert-VITS2


In [3]:
# faster-whisperをインストール
!pip install faster-whisper

# transformersを互換性のあるバージョンにダウングレード
!pip install transformers==4.30.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.8/38.8 MB 23.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 23.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 81.4 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproje

## 2. 学習に使うデータ準備

すでに音声ファイル（1ファイル2-12秒程度）とその書き起こしデータがある場合は2.2を、ない場合は2.1を実行してください。

### 2.1 音声ファイルからのデータセットの作成（ある人はスキップ可）

音声ファイル（1ファイル2-12秒程度）とその書き起こしのデータセットを持っていない方は、（日本語の）音声ファイルのみから以下の手順でデータセットを作成することができます。Google drive上の`Style-Bert-VITS2/inputs/`フォルダに音声ファイル（wavやmp3等の通常の音声ファイル形式、1ファイルでも複数ファイルでも可）を置いて、下を実行すると、データセットが作られ、自動的に正しい場所へ配置されます。

**2024-06-02のVer 2.5以降**、`inputs/`フォルダにサブフォルダを2個以上作ってそこへ音声ファイルをスタイルに応じて振り分けて置くと、学習の際にサブディレクトリに応じたスタイルが自動的に作成されます。デフォルトスタイルのみでよい場合や手動でスタイルを後で作成する場合は`inputs/`直下へ入れれば大丈夫です。

In [6]:
# 元となる音声ファイル（wav形式）を入れるディレクトリ
input_dir = "/content/drive/MyDrive/Style-Bert-VITS2/inputs"
# モデル名（話者名）を入力
model_name = "shinozaki_ai"

# こういうふうに書き起こして欲しいという例文（句読点の入れ方・笑い方や固有名詞等）
initial_prompt = "こんにちは。元気、ですかー？ふふっ、私は……ちゃんと元気だよ！"

!python slice.py -i {input_dir} --model_name {model_name}
#!python transcribe.py --model_name {model_name} --initial_prompt {initial_prompt} --use_hf_whisper
!python transcribe.py --model_name {model_name} --initial_prompt {initial_prompt}

01-21 02:14:55 |  INFO  | slice.py:167 | Found 1 audio files.
01-21 02:14:55 |WARNING | slice.py:169 | Output directory /content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/raw already exists, deleting...
Downloading: "https://github.com/litagin02/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
100%|##########| 1/1 [00:15<00:00, 15.58s/it]
01-21 02:15:14 |  INFO  | slice.py:265 | Slice done! Total time: 8.07 min, 82 files.
01-21 02:15:18 |  INFO  | transcribe.py:157 | Found 82 WAV files
01-21 02:15:18 |WARNING | transcribe.py:163 | /content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/esd.list exists, backing up to /content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/esd.list.bak
01-21 02:15:18 |WARNING | transcribe.py:166 | /content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/esd.list.bak exists, deleting...
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated.

成功したらそのまま3へ進んでください

### 2.2 音声ファイルと書き起こしデータがすでにある場合

指示に従って適切にデータセットを配置してください。

次のセルを実行して、学習データをいれるフォルダ（1で設定した`dataset_root`）を作成します。

In [5]:
import os

os.makedirs(dataset_root, exist_ok=True)

まず音声データと、書き起こしテキストを用意してください。

それを次のように配置します。
```
├── Data/
│   ├── {モデルの名前}
│   │   ├── esd.list
│   │   ├── raw/
│   │   │   ├── foo.wav
│   │   │   ├── bar.mp3
│   │   │   ├── style1/
│   │   │   │   ├── baz.wav
│   │   │   │   ├── qux.wav
│   │   │   ├── style2/
│   │   │   │   ├── corge.wav
│   │   │   │   ├── grault.wav
...
```

### 配置の仕方
- 上のように配置すると、`style1/`と`style2/`フォルダの内部（直下以外も含む）に入っている音声ファイルたちから、自動的にデフォルトスタイルに加えて`style1`と`style2`というスタイルが作成されます
- 特にスタイルを作る必要がない場合や、スタイル分類機能等でスタイルを作る場合は、`raw/`フォルダ直下に全てを配置してください。このように`raw/`のサブディレクトリの個数が0または1の場合は、スタイルはデフォルトスタイルのみが作成されます。
- 音声ファイルのフォーマットはwav形式以外にもmp3等の多くの音声ファイルに対応しています

### 書き起こしファイル`esd.list`

`Data/{モデルの名前}/esd.list` ファイルには、以下のフォーマットで各音声ファイルの情報を記述してください。


```
path/to/audio.wav(wavファイル以外でもこう書く)|{話者名}|{言語ID、ZHかJPかEN}|{書き起こしテキスト}
```

- ここで、最初の`path/to/audio.wav`は、`raw/`からの相対パスです。つまり、`raw/foo.wav`の場合は`foo.wav`、`raw/style1/bar.wav`の場合は`style1/bar.wav`となります。
- 拡張子がwavでない場合でも、`esd.list`には`wav`と書いてください、つまり、`raw/bar.mp3`の場合でも`bar.wav`と書いてください。


例：
```
foo.wav|hanako|JP|こんにちは、元気ですか？
bar.wav|taro|JP|はい、聞こえています……。何か用ですか？
style1/baz.wav|hanako|JP|今日はいい天気ですね。
style1/qux.wav|taro|JP|はい、そうですね。
...
english_teacher.wav|Mary|EN|How are you? I'm fine, thank you, and you?
...
```
もちろん日本語話者の単一話者データセットでも構いません。

## 3. 学習の前処理

次に学習の前処理を行います。必要なパラメータをここで指定します。次のセルに設定等を入力して実行してください。「～～かどうか」は`True`もしくは`False`を指定してください。

In [7]:
# 上でつけたフォルダの名前`Data/{model_name}/`
model_name = "shinozaki_ai"

# JP-Extra （日本語特化版）を使うかどうか。日本語の能力が向上する代わりに英語と中国語は使えなくなります。
use_jp_extra = True

# 学習のバッチサイズ。VRAMのはみ出具合に応じて調整してください。
batch_size = 4

# 学習のエポック数（データセットを合計何周するか）。
# 100で多すぎるほどかもしれませんが、もっと多くやると質が上がるのかもしれません。
epochs = 100

# 保存頻度。何ステップごとにモデルを保存するか。分からなければデフォルトのままで。
save_every_steps = 1000

# 音声ファイルの音量を正規化するかどうか
normalize = False

# 音声ファイルの開始・終了にある無音区間を削除するかどうか
trim = False

# 読みのエラーが出た場合にどうするか。
# "raise"ならテキスト前処理が終わったら中断、"skip"なら読めない行は学習に使わない、"use"なら無理やり使う
yomi_error = "skip"

上のセルが実行されたら、次のセルを実行して学習の前処理を行います。

In [8]:
from gradio_tabs.train import preprocess_all
from style_bert_vits2.nlp.japanese import pyopenjtalk_worker


pyopenjtalk_worker.initialize_worker()

preprocess_all(
    model_name=model_name,
    batch_size=batch_size,
    epochs=epochs,
    save_every_steps=save_every_steps,
    num_processes=2,
    normalize=normalize,
    trim=trim,
    freeze_EN_bert=False,
    freeze_JP_bert=False,
    freeze_ZH_bert=False,
    freeze_style=False,
    freeze_decoder=False,
    use_jp_extra=use_jp_extra,
    val_per_lang=0,
    log_interval=200,
    yomi_error=yomi_error,
)

01-21 02:17:46 | DEBUG  | __init__.py:92 | try starting pyopenjtalk worker server
01-21 02:17:47 | DEBUG  | __init__.py:130 | pyopenjtalk worker server started
01-21 02:17:47 |  INFO  | train.py:72 | Step 1: start initialization...
model_name: shinozaki_ai, batch_size: 4, epochs: 100, save_every_steps: 1000, freeze_ZH_bert: False, freeze_JP_bert: False, freeze_EN_bert: False, freeze_style: False, freeze_decoder: False, use_jp_extra: True
01-21 02:17:47 |WARNING | train.py:103 | Step 1: /content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/models already exists, so copy it to backup to /content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/models_backup
01-21 02:19:03 |SUCCESS | train.py:132 | Step 1: initialization finished.
01-21 02:19:03 |  INFO  | train.py:137 | Step 2: start resampling...
01-21 02:19:03 |  INFO  | subprocess.py:23 | Running: resample.py -i /content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/raw -o /content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_

(True, 'Success: 全ての前処理が完了しました。ターミナルを確認しておかしいところがないか確認するのをおすすめします。')

In [9]:
# preprocess_text.pyを手動で実行して、詳細なエラーメッセージを確認
!python preprocess_text.py \
  --config-path /content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/config.json \
  --transcription-path /content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/esd.list \
  --train-path /content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/train.list \
  --val-path /content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/val.list \
  --val-per-lang 0 \
  --yomi_error skip \
  --correct_path \
  --use_jp_extra

01-21 02:22:17 | DEBUG  | __init__.py:130 | pyopenjtalk worker server started
  0%|          | 0/82 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
01-21 02:22:19 |  INFO  | bert_models.py:178 | Loaded the JP BERT tokenizer from /content/Style-Bert-VITS2/bert/deberta-v2-large-japanese-char-wwm
10

## 4. 学習

前処理が正常に終わったら、学習を行います。次のセルを実行すると学習が始まります。

学習の結果は、上で指定した`save_every_steps`の間隔で、Google Driveの中の`Style-Bert-VITS2/Data/{モデルの名前}/model_assets/`フォルダに保存されます。

このフォルダをダウンロードし、ローカルのStyle-Bert-VITS2の`model_assets`フォルダに上書きすれば、学習結果を使うことができます。

In [10]:
# 上でつけたモデル名を入力。学習を途中からする場合はきちんとモデルが保存されているフォルダ名を入力。
model_name = "shinozaki_ai"


import yaml
from gradio_tabs.train import get_path

paths = get_path(model_name)
dataset_path = str(paths.dataset_path)
config_path = str(paths.config_path)

with open("default_config.yml", "r", encoding="utf-8") as f:
    yml_data = yaml.safe_load(f)
yml_data["model_name"] = model_name
with open("config.yml", "w", encoding="utf-8") as f:
    yaml.dump(yml_data, f, allow_unicode=True)

In [11]:
import numpy as np

# 空のスタイルベクトルを作成（デフォルトスタイル用）
style_vectors = np.zeros((1, 256), dtype=np.float32)
np.save("/content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/style_vectors.npy", style_vectors)
print("style_vectors.npy を作成しました")

# 確認
import os
if os.path.exists("/content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/style_vectors.npy"):
    print("✓ ファイルが正しく作成されました")
else:
    print("✗ ファイル作成に失敗しました")

style_vectors.npy を作成しました
✓ ファイルが正しく作成されました


In [12]:
# config.jsonに手動でスタイル情報を追加
import json

config_path = "/content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/config.json"

with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

# スタイル情報を追加（デフォルトスタイルのみ）
config["data"]["num_styles"] = 1
config["data"]["style2id"] = {"Neutral": 0}

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("✓ config.jsonにスタイル情報を追加しました")

✓ config.jsonにスタイル情報を追加しました


In [15]:
# 日本語特化版を「使う」場合
#!python train_ms_jp_extra.py --config {config_path} --model {dataset_path} --assets_root {assets_root}

!python train_ms_jp_extra.py --config {config_path} --model {dataset_path} --assets_root {assets_root} --skip_default_style

2026-01-21 02:22:53.515151: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768962173.754854    5612 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768962173.819569    5612 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768962174.333238    5612 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768962174.333292    5612 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768962174.333299    5612 computation_placer.cc:177] computation placer alr

In [ ]:
# 最新のモデル（G_2200.pth）をsafetensors形式に変換してエクスポート
!python export_model.py --model /content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/models/G_2200.pth

In [ ]:
# 日本語特化版を「使わない」場合
!python train_ms.py --config {config_path} --model {dataset_path} --assets_root {assets_root}

In [19]:
# config.jsonをmodel_assetsフォルダにコピー
import shutil

source = "/content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/config.json"
dest_dir = "/content/drive/MyDrive/Style-Bert-VITS2/model_assets/shinozaki_ai/"

# 念のためフォルダが存在するか確認
import os
os.makedirs(dest_dir, exist_ok=True)

# コピー
shutil.copy(source, dest_dir)
print(f"✓ config.jsonをコピーしました: {dest_dir}config.json")

# 確認
if os.path.exists(f"{dest_dir}config.json"):
    print("✓ ファイルが正しく配置されました")
else:
    print("✗ コピーに失敗しました")

✓ config.jsonをコピーしました: /content/drive/MyDrive/Style-Bert-VITS2/model_assets/shinozaki_ai/config.json
✓ ファイルが正しく配置されました


In [21]:
# style_vectors.npyをmodel_assetsフォルダにコピー
import shutil
import os

source = "/content/drive/MyDrive/Style-Bert-VITS2/Data/shinozaki_ai/style_vectors.npy"
dest_dir = "/content/drive/MyDrive/Style-Bert-VITS2/model_assets/shinozaki_ai/"

# コピー
shutil.copy(source, dest_dir)
print(f"✓ style_vectors.npyをコピーしました: {dest_dir}style_vectors.npy")

# 確認
if os.path.exists(f"{dest_dir}style_vectors.npy"):
    print("✓ ファイルが正しく配置されました")

    # model_assets/shinozaki_ai/内のファイル一覧を表示
    files = os.listdir(dest_dir)
    print(f"\nmodel_assets/shinozaki_ai/内のファイル:")
    for f in files:
        print(f"  - {f}")
else:
    print("✗ コピーに失敗しました")

✓ style_vectors.npyをコピーしました: /content/drive/MyDrive/Style-Bert-VITS2/model_assets/shinozaki_ai/style_vectors.npy
✓ ファイルが正しく配置されました

model_assets/shinozaki_ai/内のファイル:
  - shinozaki_ai_e46_s1000.safetensors
  - shinozaki_ai_e91_s2000.safetensors
  - shinozaki_ai_e100_s2200.safetensors
  - config.json
  - style_vectors.npy


In [ ]:
# 学習結果を試す・マージ・スタイル分けはこちらから
!python app.py --share

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
01-21 03:43:11 | DEBUG  | __init__.py:130 | pyopenjtalk worker server started
2026-01-21 03:43:22.130074: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has a

In [ ]:
# ONNX変換は、変換したいsafetensorsファイルを指定してこのセルを実行してください。
!python convert_onnx.py --model "Data/your_model/your_model_e100_s10000.safetensors"